In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
import datetime
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

In [4]:
df = pd.read_csv('/content/main_data_goal.csv')
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (20000, 27)


,Income,Age,Dependents,Occupation,City_Tier,Rent,Loan_Repayment,Insurance,Groceries,Transport,...,Desired_Savings,Disposable_Income,Potential_Savings_Groceries,Potential_Savings_Transport,Potential_Savings_Eating_Out,Potential_Savings_Entertainment,Potential_Savings_Utilities,Potential_Savings_Healthcare,Potential_Savings_Education,Potential_Savings_Miscellaneous
0,44637.249636,49,0,Self_Employed,Tier_1,13391.174891,0.000000,2206.490129,6658.768341,2636.970696,...,6200.537192,11265.627707,1685.696222,328.895281,465.769172,195.151320,678.292859,67.682471,0.000000,85.735517
1,26858.596592,34,2,Retired,Tier_2,5371.719318,0.000000,869.522617,2818.444460,1543.018778,...,1923.176434,9676.818733,540.306561,119.347139,141.866089,234.131168,286.668408,6.603212,56.306874,97.388606
2,50367.605084,35,1,Student,Tier_3,7555.140763,4612.103386,2201.800050,6313.222081,3221.396403,...,7050.360422,13891.450624,1466.073984,473.549752,410.857129,459.965256,488.383423,7.290892,106.653597,138.542422
3,101455.600247,21,0,Self_Employed,Tier_3,15218.340037,6809.441427,4889.418087,14690.149363,7106.130005,...,16694.965136,31617.953615,1875.932770,762.020789,1241.017448,320.190594,1389.815033,193.502754,0.000000,296.041183
4,24875.283548,52,4,Professional,Tier_2,4975.056710,3112.609398,635.907170,3034.329665,1276.155163,...,1874.099434,6265.700532,788.953124,68.160766,61.712505,187.173750,194.117130,47.294591,67.388120,96.557076


In [5]:
features = ['Income', 'Age', 'Dependents', 'Rent', 'Loan_Repayment',
            'Insurance', 'Groceries', 'Transport', 'Eating_Out',
            'Entertainment', 'Utilities', 'Healthcare', 'Education',
            'Miscellaneous']

target = 'Desired_Savings'

X = df[features]
y = df[target]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y,
                                                    test_size=0.2,
                                                    random_state=42)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

Training samples: 16000
Test samples: 4000


In [6]:
class GoalPredictionCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        val_mae = logs.get('val_mae')
        if val_mae and val_mae < 0.02:
            self.model.stop_training = True

# Functional API Model
inputs = tf.keras.Input(shape=(X_train.shape[1],), name="input_features")

x = tf.keras.layers.Dense(128, activation='relu')(inputs)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(1, activation='linear', name="desired_savings")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs, name="ArthaGoal_Prediction_Model")

model.compile(optimizer='adam',
              loss='mse',
              metrics=['mae'])

model.summary()

Model: "ArthaGoal_Prediction_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_features (InputLayer)     │ (None, 14)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         1,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ desired_savings (Dense)         │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,241 (40.00 KB)

 Trainable params: 10,241 (40.00 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
log_dir = "logs/goal/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

# Training
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_data=(X_test, y_test),
    callbacks=[tensorboard_callback, GoalPredictionCallback()]
)

Epoch 1/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 73928112.0000 - mae: 4751.5366 - val_loss: 60773756.0000 - val_mae: 3923.1343
Epoch 2/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 22973692.0000 - mae: 2728.6067 - val_loss: 10479280.0000 - val_mae: 1916.2374
Epoch 3/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 10002993.0000 - mae: 1829.2344 - val_loss: 9164991.0000 - val_mae: 1711.3451
Epoch 4/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 8704601.0000 - mae: 1619.2992 - val_loss: 7538987.0000 - val_mae: 1433.2371
Epoch 5/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 7211681.0000 - mae: 1404.0173 - val_loss: 6608748.5000 - val_mae: 1268.0358
Epoch 6/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 6761743.5000 - mae: 1315.7625 - val_loss: 6357788.5000 - val_mae: 1197.9054
Epoch 7/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 6780469.0000 - mae: 1272.3879 - val_loss: 6152525.0000 - val_mae: 1161.0016
Epoch 8/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 

In [8]:
loss, mae = model.evaluate(X_test, y_test)
print(f"\nTest MAE: {mae:.4f}")
print(f"Test Loss (MSE): {loss:.4f}")

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 4860694.0000 - mae: 1034.8921

Test MAE: 1034.8921
Test Loss (MSE): 4860694.0000


In [9]:
model.save('goal_prediction_model.keras')
joblib.dump(scaler, 'goal_scaler.pkl')

['goal_scaler.pkl']

In [10]:
def predict_goal(target_amount, current_amount, monthly_income, monthly_expenses):
    monthly_saving = monthly_income - monthly_expenses
    if monthly_saving <= 0:
        return {"error": "Tabungan bulanan harus lebih besar dari pengeluaran"}

    remaining = target_amount - current_amount
    months_needed = np.ceil(remaining / monthly_saving)

    today = datetime.datetime.now()
    predicted_date = today + datetime.timedelta(days=int(months_needed * 30))

    progress = (current_amount / target_amount) * 100

    return {
        "target": target_amount,
        "current": current_amount,
        "progress": round(progress, 1),
        "monthly_saving": round(monthly_saving),
        "months_needed": int(months_needed),
        "predicted_date": predicted_date.strftime("%B %Y"),
        "days_left": int(months_needed * 30),
        "message": f"- {int(months_needed)} bulan lagi",
        "suggestion": f"Tabung min Rp {round(monthly_saving * 1.2):,} /bln untuk lebih cepat"
    }

# Test
result = predict_goal(
    target_amount=15000000,
    current_amount=10000,
    monthly_income=100000,
    monthly_expenses=50000
)

print(result['message'])
print(result['suggestion'])

- 300 bulan lagi
Tabung min Rp 60,000 /bln untuk lebih cepat


In [11]:
import numpy as np
import tensorflow as tf
import joblib

loaded_model = tf.keras.models.load_model('goal_prediction_model.keras')
loaded_scaler = joblib.load('goal_scaler.pkl')

def prediksi_ai_arthagoal(input_fitur):
    input_array = np.array([input_fitur])

    input_scaled = loaded_scaler.transform(input_array)

    hasil_prediksi = loaded_model.predict(input_scaled)

    return hasil_prediksi[0][0]


data_user_baru = [45000, 35, 1, 10000, 0, 2000, 5000, 2000, 500, 200, 500, 100, 0, 100]

hasil = prediksi_ai_arthagoal(data_user_baru)
print(f"Prediksi Desired Savings menggunakan AI: {hasil:.2f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
Prediksi Desired Savings menggunakan AI: 1726.00


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [13]:
import os

inference_code = """
import numpy as np
import tensorflow as tf
import joblib
import pandas as pd

class ArthaGoalPredictor:
    def __init__(self, model_path='goal_prediction_model.keras', scaler_path='goal_scaler.pkl'):
        self.model = tf.keras.models.load_model(model_path)
        self.scaler = joblib.load(scaler_path)
        self.features = ['Income', 'Age', 'Dependents', 'Rent', 'Loan_Repayment',
                        'Insurance', 'Groceries', 'Transport', 'Eating_Out',
                        'Entertainment', 'Utilities', 'Healthcare', 'Education',
                        'Miscellaneous']

    def predict(self, data_list):
        df_input = pd.DataFrame([data_list], columns=self.features)
        input_scaled = self.scaler.transform(df_input)
        prediction = self.model.predict(input_scaled, verbose=0)
        return float(prediction[0][0])

# Test
if __name__ == '__main__':
    predictor = ArthaGoalPredictor()
    sample = [45000, 35, 1, 10000, 0, 2000, 5000, 2000, 500, 200, 500, 100, 0, 100]
    result = predictor.predict(sample)
    print(f'Prediksi Desired Savings: Rp {result:,.2f}')
"""

with open('/content/inference.py', 'w') as f:
    f.write(inference_code)

print(" inference.py done")

 inference.py done
